# Level 1 tracked demo - both VisDrone models, side by side

Couldn't get a confident visual verdict between `visdrone-yolov8x` and `visdrone-yolov11s`
from the benchmark notebook's per-frame exports (too cluttered - fixed large label size
meant for a full 3840px frame). Producing a proper **tracked** demo for both here instead:
SAHI-tiled detection (correct - avoids the no-tiling misclassification found earlier) fed into
`supervision`'s ByteTrack for persistent IDs, with a controllable/small label size this time.

First ~10s (300 frames) of the clip, both models, so the app can show either and you can
actually look and decide - not force a single choice from ambiguous still-frame exports.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}:", torch.cuda.get_device_name(i))

# Capture the base image's already-correctly-matched numpy/scipy versions before any pip
# install touches them - same fix pattern as the torch/torchvision pin in kaggle_starter.ipynb
# (see KAGGLE.md bug table). Adding boxmot to the install line previously let pip bump numpy
# to a version scipy's compiled internals didn't expect, breaking `import supervision`
# (confirmed via a real failed run, not a guess).
import numpy
PINS = f'"numpy=={numpy.__version__}"'
try:
    import scipy
    PINS += f' "scipy=={scipy.__version__}"'
except ImportError:
    pass
print("Pinning for all subsequent installs:", PINS)

In [ ]:
!pip install -q ultralytics sahi huggingface_hub supervision {PINS}
# boxmot==22.0.0 (the version tested locally, whose module layout our code targets) requires
# numpy>=2.2.0 - directly conflicts with the numpy pin above (confirmed via package metadata,
# not a guess). --no-deps keeps boxmot from touching numpy/scipy at all; its other real
# dependencies (checked via `pip show boxmot` / importlib.metadata.requires, not guessed) are
# installed separately below, still numpy-pinned. gdown/torch/torchvision/opencv-python/
# huggingface-hub/pandas are already covered by earlier installs or the base image; we also
# bypass boxmot's own gdown-based ReID-weight downloader entirely (see make_tracker() below),
# so gdown's exact version doesn't matter here.
!pip install -q --no-deps "boxmot==22.0.0"
!pip install -q filterpy lapx scikit-learn timm click rich {PINS}

In [ ]:
import glob
import os

# Parameterized: which fixed clip + which model(s)/tracker to run. Uses the small dedicated
# clips-compressed dataset (prithvirajgotepatil/flytbase-hackathon-clips-compressed) - faster
# mount than the original clips dataset, and the standard reusable set going forward (see
# PLAN.md). Edit these lines only for any future ad hoc run - nothing else needs to change.
SOURCE_VIDEO = "multiroad"             # or "intersection"
DURATION = "10s"                       # "10s" or "90s" available in the compressed dataset
MODELS_TO_RUN = ["visdrone-yolov11s"]  # subset ok, e.g. just one
TRACKER_TYPE = "botsort"               # "bytetrack" or "botsort" - see meta.json/output naming below

print("Contents of /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(" ", os.path.join(root, f))

found = glob.glob(f"/kaggle/input/**/{SOURCE_VIDEO}_{DURATION}.MP4", recursive=True)
assert found, f"No {SOURCE_VIDEO}_{DURATION}.MP4 found - see directory listing above"
video_path = found[0]
print("\nUsing video:", video_path)
print("Running models:", MODELS_TO_RUN, "| tracker:", TRACKER_TYPE)

## Class mapping

VisDrone's native id->name, confirmed from the benchmark run's own `model.names` output -
not guessed. Mapped to the Level 1 taxonomy per `pipeline/taxonomy.py` (kept in sync manually
since this notebook runs standalone on Kaggle, not importing local repo files).

In [ ]:
VISDRONE_NAMES = {0: "pedestrian", 1: "people", 2: "bicycle", 3: "car", 4: "van",
                   5: "truck", 6: "tricycle", 7: "awning-tricycle", 8: "bus", 9: "motor", 10: "others"}
VISDRONE_TO_TAXONOMY = {
    "car": "car", "van": "LGV_candidate", "truck": "HGV_candidate", "bus": "bus",
    "motor": "motorcycle", "pedestrian": "pedestrian", "people": "pedestrian",
    "bicycle": "other", "tricycle": "other", "awning-tricycle": "other", "others": "other",
}

## Run: SAHI-tiled detection -> supervision ByteTrack -> annotated video + CSV, per model

Label size set explicitly small (`text_scale=0.4`) - the benchmark notebook's illegibility
problem was purely a visualization default, not a detection quality issue.

In [ ]:
import csv
import json
import time

import cv2
import numpy as np
import supervision as sv
from huggingface_hub import hf_hub_download
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

ALL_MODEL_REPOS = {
    "visdrone-yolov8x": "dronefreak/visdrone-yolov8x",
    "visdrone-yolov11s": "dronefreak/visdrone-yolov11s",
}
CANDIDATES = {
    name: hf_hub_download(repo_id=ALL_MODEL_REPOS[name], filename="best.pt")
    for name in MODELS_TO_RUN
}
# Raised from 0.15 (the benchmark's exploratory counting threshold) after the first tracked
# run showed severe track-ID churn (1600+ unique tracks in a 10s clip) - low-confidence,
# tile-boundary-flicker detections were spawning and killing tracks almost every frame.
CONFIDENCE_THRESHOLD = 0.35
ROOT_OUTPUT_DIR = f"/kaggle/working/tracked_demo/{SOURCE_VIDEO}/{DURATION}"
# Suffix keeps ByteTrack outputs from earlier runs untouched - never overwritten by this.
TRACKER_SUFFIX = "" if TRACKER_TYPE == "bytetrack" else f"-{TRACKER_TYPE}"

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.4, text_thickness=1, text_padding=2)


def make_tracker():
    if TRACKER_TYPE == "bytetrack":
        return sv.ByteTrack()
    if TRACKER_TYPE == "botsort":
        import gdown
        import boxmot
        from boxmot.trackers.bbox.botsort import BotSort
        from boxmot.reid.core.reid import ReID

        # boxmot's own ReID-weight auto-download calls gdown.download(..., fuzzy=...), an
        # argument our installed gdown doesn't have (confirmed locally - a real version
        # mismatch, not a guess). Pre-fetch the weight file ourselves so boxmot finds it
        # already in place and skips that broken code path entirely. See PLAN.md.
        weights_dir = os.path.join(os.path.dirname(boxmot.__file__), "..", "models")
        os.makedirs(weights_dir, exist_ok=True)
        weights_path = os.path.join(weights_dir, "osnet_x0_25_msmt17.pt")
        if not os.path.exists(weights_path):
            gdown.download(id="1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF", output=weights_path, quiet=False)

        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        reid = ReID(path=weights_path, device=device, half=torch.cuda.is_available())
        # track_buffer raised from ByteTrack's default (30 frames / ~1s) to 60 (~2s) - the
        # cheap free tune from PLAN.md's occlusion-fix list, applied alongside the appearance
        # ReID upgrade rather than instead of it.
        return BotSort(reid_model=reid.model, track_buffer=60, frame_rate=30, with_reid=True)
    raise ValueError(f"Unknown TRACKER_TYPE: {TRACKER_TYPE}")


for name, weights_path in CANDIDATES.items():
    output_name = f"{name}{TRACKER_SUFFIX}"
    print(f"\n=== {output_name} ===")
    # Segregated per source/duration/model+tracker: nothing shared/flat-namespaced,
    # and existing ByteTrack folders are never touched by a BoT-SORT run.
    model_dir = f"{ROOT_OUTPUT_DIR}/{output_name}"
    os.makedirs(model_dir, exist_ok=True)

    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics", model_path=weights_path,
        confidence_threshold=CONFIDENCE_THRESHOLD, device="cuda:0" if torch.cuda.is_available() else "cpu",
    )
    tracker = make_tracker()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_frames_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    writer = cv2.VideoWriter(f"{model_dir}/video_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    rows = []
    t0 = time.time()
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        result = get_sliced_prediction(
            frame, detection_model, slice_height=768, slice_width=768,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0,
        )
        preds = result.object_prediction_list

        if TRACKER_TYPE == "bytetrack":
            if preds:
                xyxy = np.array([p.bbox.to_xyxy() for p in preds], dtype=np.float32)
                confidence = np.array([p.score.value for p in preds], dtype=np.float32)
                class_id = np.array([p.category.id for p in preds], dtype=int)
                detections = sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_id)
            else:
                detections = sv.Detections.empty()
            tracked = tracker.update_with_detections(detections)
            track_rows = list(zip(tracked.xyxy, tracked.class_id, tracked.tracker_id, tracked.confidence))
        else:  # botsort - dets shape (N,6): x1,y1,x2,y2,conf,cls (confirmed via local smoke test)
            if preds:
                dets = np.array(
                    [[*p.bbox.to_xyxy(), p.score.value, p.category.id] for p in preds], dtype=np.float32
                )
            else:
                dets = np.empty((0, 6), dtype=np.float32)
            tracked_arr = np.asarray(tracker.update(dets, frame))
            # boxmot output columns: x1,y1,x2,y2,id,conf,cls,det_ind (confirmed via local test)
            track_rows = [(row[:4], row[6], row[4], row[5]) for row in tracked_arr]

        labels = []
        for xyxy_box, cid, tid, conf in track_rows:
            native = VISDRONE_NAMES.get(int(cid), "others")
            taxonomy = VISDRONE_TO_TAXONOMY.get(native, "other")
            labels.append(f"#{int(tid)} {taxonomy}")
            x1, y1, x2, y2 = [float(v) for v in xyxy_box]
            rows.append({
                "frame": frame_idx, "time_s": round(frame_idx / fps, 2),
                "track_id": int(tid), "native_class": native, "taxonomy_class": taxonomy,
                "confidence": round(float(conf), 3),
                "x1": round(x1, 1), "y1": round(y1, 1), "x2": round(x2, 1), "y2": round(y2, 1),
            })

        if TRACKER_TYPE == "bytetrack":
            annotated = box_annotator.annotate(frame.copy(), tracked)
            annotated = label_annotator.annotate(annotated, tracked, labels=labels)
        else:
            annotated = frame.copy()
            for (xyxy_box, _cid, _tid, _conf), label in zip(track_rows, labels):
                x1, y1, x2, y2 = [int(v) for v in xyxy_box]
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(annotated, label, (x1, max(0, y1 - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
        writer.write(annotated)

        frame_idx += 1
        if frame_idx % 50 == 0:
            print(f"  frame {frame_idx}/{n_frames_total} ({time.time() - t0:.0f}s elapsed)")

    cap.release()
    writer.release()

    fieldnames = ["frame", "time_s", "track_id", "native_class", "taxonomy_class",
                  "confidence", "x1", "y1", "x2", "y2"]
    with open(f"{model_dir}/detections.csv", "w", newline="") as f:
        w_csv = csv.DictWriter(f, fieldnames=fieldnames)
        w_csv.writeheader()
        w_csv.writerows(rows)

    n_tracks = len(set(r["track_id"] for r in rows))
    meta = {
        "source_video": SOURCE_VIDEO,
        "duration_s": int(DURATION.rstrip("s")),
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "model": output_name,
        "weights_source": ALL_MODEL_REPOS[name],
        "frames_processed": frame_idx,
        "fps": fps,
        "frame_width": w,
        "frame_height": h,
        "detection_rows": len(rows),
        "unique_tracks": n_tracks,
        "tracker": "supervision.ByteTrack" if TRACKER_TYPE == "bytetrack" else "boxmot.BotSort (+ReID, track_buffer=60)",
        "sliced_inference": {"slice_size": 768, "overlap": 0.2, "tool": "SAHI"},
    }
    with open(f"{model_dir}/meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    with open(f"{model_dir}/detections.json", "w") as f:
        json.dump(rows, f, indent=2)

    print(f"{output_name}: {frame_idx} frames, {len(rows)} detection rows, {n_tracks} unique tracks")
    print(f"  all outputs under: {model_dir}/")

## Transcode to browser-compatible H.264

Same issue as the local quick-track sample: OpenCV's `mp4v`-written output often won't play
via a browser `<video>` tag. Kaggle's base image has ffmpeg - transcode here so the pulled
files are directly usable by the Streamlit app, no local transcode step needed after pulling.

In [ ]:
for name in CANDIDATES:
    output_name = f"{name}{TRACKER_SUFFIX}"
    model_dir = f"{ROOT_OUTPUT_DIR}/{output_name}"
    src = f"{model_dir}/video_raw.mp4"
    dst = f"{model_dir}/video_h264.mp4"
    os.system(f'ffmpeg -y -i "{src}" -c:v libx264 -pix_fmt yuv420p -movflags +faststart "{dst}"')
    print(output_name, "->", dst, os.path.exists(dst))

## Next

Pull `/kaggle/working/tracked_demo` via `kaggle kernels output`. Output is segregated per
model: `tracked_demo/<model_name>/{video_h264.mp4, detections.csv, detections.json, meta.json}`.
Drop each model's folder into `videos/samples/<model_name>/` and point `app.py`'s selector at
both - it reads `meta.json` to label which model produced what's on screen.